In [1]:
import os, re, csv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from typing import Dict, Tuple, List, Sequence, Union


In [2]:
# ---------- Patterns ----------
DIR_RE = re.compile(r"^d=(\d+)$")
HP_RE  = re.compile(
    r"^(?P<measure>normal|t)_(?P<transform>CDF|linear|piecewise_linear|quadratic)_n_(?P<n>\d+)(?:_k_(?P<k>-?\d+(?:\.\d+)?))?$"
)
RKHS_RE = re.compile(
    r"^rmse_(?P<measure>normal|t)_(?P<transform>CDF|linear|piecewise_linear|quadratic)_n_(?P<n>\d+)_d_(?P<d>\d+)_df_(?P<df>\d+)_testN_(?P<testN>\d+)\.csv$"
)

Key = Tuple[str, str, int, int]  # (measure, transform, n, d)

# Ns, MEASURES assumed; change defaults below if you like
DEFAULT_NS = [100, 300, 500, 1000]
DEFAULT_MEASURES = ["normal", "t"]
DEFAULT_TRANSFORMS = ["CDF", "linear", "quadratic"]


In [3]:
# ---------- CSV loader (robust to headers) ----------
def _load_numeric_series_from_csv(path: str) -> np.ndarray:
    vals: List[float] = []
    with open(path, "r", newline="") as f:
        reader = csv.reader(f)
        header = next(reader, None)
        col_idx = None
        if header:
            lower = [h.lower() if isinstance(h, str) else "" for h in header]
            # prefer likely value columns
            for name in ("test_rmse", "rmse", "l2_loss", "loss", "value"):
                if name in lower:
                    col_idx = lower.index(name)
                    break
            if col_idx is None:
                col_idx = 1 if len(lower) > 1 else 0
        for row in reader:
            if not row:
                continue
            cell = row[col_idx] if (col_idx is not None and col_idx < len(row)) else row[0]
            try:
                v = float(cell)
                if np.isfinite(v):
                    vals.append(v)
            except Exception:
                pass
    return np.asarray(vals, dtype=float)

# ---------- Loaders ----------
def load_original_tree(orig_root: str, exact_k_text: str | None = None) -> Dict[Key, np.ndarray]:
    data: Dict[Key, List[np.ndarray]] = {}
    for dname in os.listdir(orig_root):
        dm = DIR_RE.match(dname)
        if not dm:
            continue
        d = int(dm.group(1))
        dpath = os.path.join(orig_root, dname)
        if not os.path.isdir(dpath):
            continue
        for sub in os.listdir(dpath):
            spath = os.path.join(dpath, sub)
            if not os.path.isdir(spath):
                continue
            m = HP_RE.match(sub)
            if not m:
                continue
            if exact_k_text is not None and m.group("k") != exact_k_text:
                continue
            measure   = m.group("measure")
            transform = m.group("transform")
            n         = int(m.group("n"))
            series_list: List[np.ndarray] = []
            for fn in os.listdir(spath):
                if fn.lower().endswith(".csv"):
                    arr = _load_numeric_series_from_csv(os.path.join(spath, fn))
                    if arr.size > 0:
                        series_list.append(arr)
            if series_list:
                key: Key = (measure, transform, n, d)
                data.setdefault(key, []).extend(series_list)
    out: Dict[Key, np.ndarray] = {}
    for k, lst in data.items():
        out[k] = np.concatenate(lst, axis=0)
    return out

def load_rkhs_dir(rkhs_root: str) -> Dict[Key, np.ndarray]:
    out: Dict[Key, np.ndarray] = {}
    for root, _, files in os.walk(rkhs_root):
        for fn in files:
            m = RKHS_RE.match(fn)
            if not m:
                continue
            measure   = m.group("measure")
            transform = m.group("transform")
            n         = int(m.group("n"))
            d         = int(m.group("d"))
            key: Key  = (measure, transform, n, d)
            vals = _load_numeric_series_from_csv(os.path.join(root, fn))
            if vals.size > 0:
                out[key] = vals
    return out


In [8]:
def _pretty_title(measure: str, transform: str, t_df_label: int | None):
    # Measure pretty
    if measure == "normal":
        meas_str = "Normal"
    elif measure == "t":
        meas_str = f"t({t_df_label})" if t_df_label is not None else "t"
    else:
        meas_str = measure

    # Transform pretty
    trans_map = {
        "CDF": "CDF",
        "linear": "Linear",
        "quadratic": "Sign-quadratic",
    }
    trans_str = trans_map.get(transform, transform)
    return f"{meas_str}, {trans_str}"

def _make_blue_palette(k: int):
    """k distinct blues (vary lightness)."""
    base = np.array(mcolors.to_rgb("#000154"))  # matplotlib default blue
    whites = np.array(mcolors.to_rgb("#56CCFFFF"))
    if k <= 1:
        return ["#1f77b4"]
    # mix with white to get lighter variants
    factors = np.linspace(0.0, 1.0, k)  # 0 -> base, up to lighter
    cols = []
    for f in factors:
        rgb = (1 - f) * base + f * whites
        cols.append(mcolors.to_hex(rgb, keep_alpha=False))
    return cols

def plot_dim_grid_side_by_side_multi(
    orig_roots: list[str],
    rkhs_root: str,
    d_target: int,
    transforms=None,                         # e.g. ["CDF","linear","quadratic"]
    measures=None,                           # e.g. ["normal","t"]
    Ns=None,                                 # e.g. [100,300,500,1000]
    t_df_label: int | None = 6,
    orig_labels: list[str] | None = None,
    rkhs_label: str = "Oracle kernel",
    legend_loc: str = "upper right",
    out_path: str | None = None,
    # NEW: widen the spacing between *groups* of sample sizes
    group_gap: float = 1.0,                  # 1.0 = old spacing; try 1.6, 2.0, ...
    # NEW: clip large values
    upper_clip: float = 50.0,                # drop any values > upper_clip
    # Keep intra-group spacing the same
    delta: float = 0.22,                      # spacing between estimators within the same group
    log_y_axis: bool = True                    
):
    """
    Multiple 'original' estimators (one per orig_root) vs a single RKHS series.
    For each sample-size n, draws a centered group of (M originals + 1 RKHS) boxplots.

    New:
      - group_gap: multiplies the distance between neighboring sample-size groups
      - upper_clip: exclude values > threshold (and <=0 are dropped for log safety)
    """
    assert isinstance(orig_roots, (list, tuple)) and len(orig_roots) >= 1

    # --- use your existing loaders ---
    orig_maps = [load_original_tree(root) for root in orig_roots]
    rkhs_map  = load_rkhs_dir(rkhs_root)

    measures   = list(measures   or DEFAULT_MEASURES)
    transforms = list(transforms or DEFAULT_TRANSFORMS)
    Ns         = list(Ns         or DEFAULT_NS)

    nrows, ncols = len(measures), len(transforms)
    fig_w, fig_h = 4.8 * ncols, 4.1 * nrows
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), sharey=False)

    # normalize axes to 2D
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes[np.newaxis, :]
    elif ncols == 1:
        axes = axes[:, np.newaxis]

    blue_cols = _make_blue_palette(len(orig_maps))
    red_col   = "red"
    width     = 0.75 * delta

    if orig_labels is None:
        orig_labels = [f"Neural network {i+1}" for i in range(len(orig_maps))]
    else:
        assert len(orig_labels) == len(orig_maps)

    # Precompute x positions for group centers with enlarged gap
    centers = [1.0 + (i * group_gap) for i in range(len(Ns))]

    drew_legend = False

    for r, measure in enumerate(measures):
        for c, transform in enumerate(transforms):
            ax = axes[r, c]

            for idx, (n, base) in enumerate(zip(Ns, centers)):
                # symmetric offsets for (M originals + 1 RKHS)
                M = len(orig_maps)
                offsets = np.linspace(-M/2, M/2, M + 1) * delta

                # --- RKHS ---
                k_vals = rkhs_map.get((measure, transform, n, d_target), None)
                if k_vals is not None and k_vals.size > 0:
                    k_vals = np.asarray(k_vals, dtype=float)
                    k_vals = k_vals[(k_vals > 0.0) & (k_vals <= upper_clip)]
                    if k_vals.size > 0:
                        posK = base + offsets[0]
                        bp2 = ax.boxplot([k_vals], positions=[posK], widths=width,
                                         showmeans=True, meanline=True, patch_artist=True)
                        for box in bp2["boxes"]:
                            box.set_facecolor(red_col)
                        for med in bp2["medians"]:
                            med.set_color("white")
                        for mean in bp2["means"]:
                            mean.set_color("white")

                # --- Originals ---
                for j, (omap, col) in enumerate(zip(orig_maps, blue_cols)):
                    key = (measure, transform, n, d_target)
                    vals = omap.get(key, None)
                    if vals is None or vals.size == 0:
                        continue
                    vals = np.asarray(vals, dtype=float)
                    vals = vals[(vals > 0.0) & (vals <= upper_clip)]
                    if vals.size == 0:
                        continue
                    pos = base + offsets[j+1]
                    bp = ax.boxplot([vals], positions=[pos], widths=width,
                                    showmeans=True, meanline=True, patch_artist=True)
                    for box in bp["boxes"]:
                        box.set_facecolor(col)
                    for med in bp["medians"]:
                        med.set_color("white")
                    for mean in bp["means"]:
                        mean.set_color("white")

            # Axes formatting
            if log_y_axis:
                ax.set_yscale("log")
            half_span = (len(orig_maps)/2) * delta + width
            xmin = centers[0]   - half_span - 0.4
            xmax = centers[-1]  + half_span + 0.4
            ax.set_xlim(xmin, xmax)
            ax.set_xticks(centers)
            ax.set_xticklabels([str(n) for n in Ns])
            ax.set_xlabel("sample size n")
            if c == 0:
                ax.set_ylabel("loss (log scale)")

            ax.set_title(_pretty_title(measure, transform, t_df_label))
            ax.grid(True, axis="y", linestyle="--", linewidth=0.6, alpha=0.6)

            # --- NEW: dashed vertical lines between sample-size groups ---
            for i in range(len(centers) - 1):
                x_mid = 0.5 * (centers[i] + centers[i+1])
                ax.axvline(x_mid, linestyle="--", color="gray", alpha=0.4, linewidth=0.8)

            if not drew_legend and r == 0 and c == 0:
                import matplotlib.patches as mpatches
                handles = [mpatches.Patch(color=red_col)] + [mpatches.Patch(color=col) for col in blue_cols]
                labels  = [rkhs_label] + orig_labels[:]
                ax.legend(handles, labels, loc=legend_loc, frameon=True)
                drew_legend = True

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    if out_path:
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
        plt.close(fig)
        print(f"[saved] {out_path}")
    else:
        plt.show()

        
def plot_row_side_by_side_multi(
    orig_roots: list[str],
    rkhs_root: str,
    d_target: int,
    transforms=None,                         # (ignored in this 4-panel layout)
    measures=None,                           # (ignored in this 4-panel layout)
    Ns=None,                                 # e.g. [100,300,500,1000]
    t_df_label: int | None = 6,
    orig_labels: list[str] | None = None,
    rkhs_label: str = "Oracle kernel",
    legend_loc: str = "upper right",
    out_path: str | None = None,
    group_gap: float = 1.0,                  # inter-group spacing (between sample sizes)
    upper_clip: float = 50.0,                # drop any values > upper_clip
    delta: float = 0.22                      # intra-group spacing (between estimators)
):
    """
    Four subplots in one row with fixed order:
      [Normal–CDF] [t(df)–CDF] [Normal–Sign-quadratic] [t(df)–Sign-quadratic]

    Multiple 'original' estimators (one per orig_root) vs a single RKHS series.
    For each sample-size n, draws a centered group of (M originals + 1 RKHS) boxplots.

    - group_gap   controls distance between different sample sizes (wider spacing).
    - upper_clip  removes values > threshold; values <= 0 are also removed (log-scale safe).
    - delta       controls distance between boxes within the same group; kept constant.
    """
    assert isinstance(orig_roots, (list, tuple)) and len(orig_roots) >= 1

    # Load data maps using your existing loaders
    orig_maps = [load_original_tree(root) for root in orig_roots]
    rkhs_map  = load_rkhs_dir(rkhs_root)

    Ns = list(Ns or DEFAULT_NS)

    # Fixed 4-panel order (measure, transform)
    pairs = [("normal", "CDF"),
             ("t",      "CDF"),
             ("normal", "quadratic"),
             ("t",      "quadratic")]

    # Figure
    fig_w, fig_h = 4.8 * 4, 4.2
    fig, axes = plt.subplots(1, 4, figsize=(fig_w, fig_h), sharey=False)
    #fig.suptitle(f"Original vs RKHS — d={d_target}", y=0.98)

    blue_cols = _make_blue_palette(len(orig_maps))
    red_col   = "red"
    width     = 0.75 * delta

    if orig_labels is None:
        orig_labels = [f"Neural network {i+1}" for i in range(len(orig_maps))]
    else:
        assert len(orig_labels) == len(orig_maps)

    # Group centers for the sample sizes with enlarged gaps
    centers = [1.0 + (i * group_gap) for i in range(len(Ns))]

    drew_legend = False

    for ax, (measure, transform) in zip(axes, pairs):
        for base, n in zip(centers, Ns):
            # symmetric offsets for (M originals + 1 RKHS)
            M = len(orig_maps)
            offsets = np.linspace(-M/2, M/2, M + 1) * delta  # last slot = RKHS

            # Originals
            for j, (omap, col) in enumerate(zip(orig_maps, blue_cols)):
                key = (measure, transform, n, d_target)
                vals = omap.get(key, None)
                if vals is None or vals.size == 0:
                    continue
                vals = np.asarray(vals, dtype=float)
                vals = vals[(vals > 0.0) & (vals <= upper_clip)]
                if vals.size == 0:
                    continue
                pos = base + offsets[j]
                bp = ax.boxplot([vals], positions=[pos], widths=width,
                                showmeans=True, meanline=True, patch_artist=True)
                for box in bp["boxes"]:
                    box.set_facecolor(col)
                for med in bp["medians"]:
                    med.set_color("white")
                for mean in bp["means"]:
                    mean.set_color("white")

            # RKHS
            k_vals = rkhs_map.get((measure, transform, n, d_target), None)
            if k_vals is not None and k_vals.size > 0:
                k_vals = np.asarray(k_vals, dtype=float)
                k_vals = k_vals[(k_vals > 0.0) & (k_vals <= upper_clip)]
                if k_vals.size > 0:
                    posK = base + offsets[-1]
                    bp2 = ax.boxplot([k_vals], positions=[posK], widths=width,
                                     showmeans=True, meanline=True, patch_artist=True)
                    for box in bp2["boxes"]:
                        box.set_facecolor(red_col)
                    for med in bp2["medians"]:
                        med.set_color("white")
                    for mean in bp2["means"]:
                        mean.set_color("white")

        # Axes formatting
        ax.set_yscale("log")
        half_span = (len(orig_maps)/2) * delta + width
        xmin = centers[0]  - half_span - 0.4
        xmax = centers[-1] + half_span + 0.4
        ax.set_xlim(xmin, xmax)
        ax.set_xticks(centers)
        ax.set_xticklabels([str(n) for n in Ns])
        ax.set_xlabel("sample size n")
        ax.set_title(_pretty_title(measure, transform, t_df_label))
        ax.grid(True, axis="y", linestyle="--", linewidth=0.6, alpha=0.6)

        # Legend only on the first subplot
        if not drew_legend:
            import matplotlib.patches as mpatches
            handles = [mpatches.Patch(color=col) for col in blue_cols]
            labels  = orig_labels[:]
            handles.append(mpatches.Patch(color=red_col))
            labels.append(rkhs_label)
            ax.legend(handles, labels, loc=legend_loc, frameon=True)
            drew_legend = True

        # y-label only on the first subplot to reduce clutter
        if ax is axes[0]:
            ax.set_ylabel("loss (log scale)")

        # If truly no data, note it
        any_data = False
        for n in Ns:
            key = (measure, transform, n, d_target)
            if rkhs_map.get(key, None) is not None:
                any_data = True
                break
            for omap in orig_maps:
                if omap.get(key, None) is not None:
                    any_data = True
                    break
            if any_data:
                break
        if not any_data:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes, alpha=0.7)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    if out_path:
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
        plt.close(fig)
        print(f"[saved] {out_path}")
    else:
        plt.show()


In [20]:
orig_roots   = [
    "ICNN_relu",
    "ICNN_requ",
    "ICNN_elu",
    #"251122_softplus_centered",
    "251125_sp_scaled_105",
    #"251125_sp_scaled_110",
    #"251125_sp_scaled_120",
]
orig_labels  = ["ReLU", "ReQU", "ELU", "SP_scaled_105"]#["ReLU", "ELU", "SP_centerd", "SP_scaled_105", "SP_scaled_110", "SP_scaled_120"]  # optional

rkhs_csv_dir = "../Hutter_estimator_code_results/ot_rkhs_rmse_results"

for dimen in [5,10,20]:
    plot_dim_grid_side_by_side_multi(
        orig_roots, rkhs_csv_dir,
        d_target=dimen,
        transforms=["CDF", "piecewise_linear", "quadratic"],  # any subset
        measures=["normal", "t"],
        Ns=[100, 300, 500, 1000],
        t_df_label=6,
        orig_labels=orig_labels,
        rkhs_label="Oracle kernel",
        legend_loc="lower left",                   # or "lower left"
        group_gap = 1.5,                  # 1.0 = old spacing; try 1.6, 2.0, ...
        upper_clip = 15.0,                # drop any values > upper_clip
        out_path=f"grid_d{dimen}.pdf", 
        log_y_axis = False
    )
    continue

    plot_row_side_by_side_multi(
        orig_roots, rkhs_csv_dir,
        d_target=dimen,
        transforms=["CDF", "quadratic"],  # any subset
        measures=["normal", "t"],
        Ns=[100, 300, 500, 1000],
        t_df_label=6,
        orig_labels=orig_labels,
        rkhs_label="Oracle kernel",
        legend_loc="lower left",                   # or "lower left"
        group_gap = 1.5,                  # 1.0 = old spacing; try 1.6, 2.0, ...
        upper_clip = 20.0,                # drop any values > upper_clip
        out_path=f"row_d{dimen}.pdf"
    )


[saved] grid_d5.pdf
[saved] grid_d10.pdf
[saved] grid_d20.pdf


In [12]:
def summarize_dimension_table(
    rkhs_root: str,
    path_nn1_root: str,
    path_nn2_root: str,
    d_target: int,
    Ns=(100, 300, 500, 1000),
    upper_clip: float | None = None,
) -> np.ndarray:
    """
    Build a 12x8 table (float) summarizing mean/SD of losses for a given dimension.
    Row order (12 rows total):
      0-3:  RKHS        -> [normal,CDF], [normal,quadratic], [t,CDF], [t,quadratic]
      4-7:  path_nn1    -> same 4 cases
      8-11: path_nn2    -> same 4 cases
    Column order (8 columns total):
      [mean@100, sd@100, mean@300, sd@300, mean@500, sd@500, mean@1000, sd@1000]

    If upper_clip is provided, values > upper_clip are excluded from the stats.
    NaNs are returned when a scenario has no data.
    """
    # Load maps
    rkhs_map  = load_rkhs_dir(rkhs_root)
    nn1_map   = load_original_tree(path_nn1_root)
    nn2_map   = load_original_tree(path_nn2_root)

    # 4 cases per estimator
    cases = [
        ("normal", "CDF"),        # N(0,1) Rank
        ("normal", "quadratic"),  # N(0,1) Sign-quadratic
        ("t",      "CDF"),        # t(6) Rank
        ("t",      "quadratic"),  # t(6) Sign-quadratic
    ]

    estimators = [rkhs_map, nn1_map, nn2_map]  # order matters
    rows = []

    def _stats(arr: np.ndarray) -> tuple[float, float]:
        if arr is None or arr.size == 0:
            return np.nan, np.nan
        vals = np.asarray(arr, dtype=float)
        # Optional clip
        if upper_clip is not None:
            vals = vals[np.isfinite(vals) & (vals <= upper_clip)]
        else:
            vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            return np.nan, np.nan
        return float(vals.mean()), float(vals.std(ddof=1)) if vals.size > 1 else (float(vals.mean()), np.nan)

    for est_map in estimators:
        for (measure, transform) in cases:
            row = []
            for n in Ns:
                arr = est_map.get((measure, transform, int(n), int(d_target)), None)
                mu, sd = _stats(arr)
                row.extend([mu, sd])
            rows.append(row)

    return np.array(rows, dtype=float)  # shape (12, 8)


def table12x8_to_latex_with_clines(
    A: np.ndarray,
    digits: int = 4,
    sample_sizes=(100, 300, 500, 1000),
    estimator_names=("Kernel", "ICNN-ReLU", "ICNN-ELU"),
    t_df_label: int = 6,
    nan_token: str = "--",
) -> str:
    """
    Convert a (12 x 8) array into a LaTeX tabular:

    \begin{tabular}{ccc|cc|cc|cc|cc}
    \hline
    & & & \multicolumn{2}{c|}{$n=N=100$} ... \\
    \hline
    & $P$ & OT map & Mean & SD & ... \\
    \hline
    <Kernel 4 rows with \cline{2-11} after the 2nd row>
    \hline
    <ICNN-ReLU 4 rows with \cline{2-11} after the 2nd row>
    \hline
    <ICNN-ELU 4 rows with \cline{2-11} after the 2nd row>
    \hline
    \end{tabular}

    Row order in A:
      0-3  : Kernel      → [(normal,CDF), (normal,quadratic), (t,CDF), (t,quadratic)]
      4-7  : ICNN-ReLU   → same 4 cases
      8-11 : ICNN-ELU    → same 4 cases

    Columns in A:
      [mean@100, sd@100, mean@300, sd@300, mean@500, sd@500, mean@1000, sd@1000]
    """
    A = np.asarray(A, dtype=float)
    if A.shape != (12, 8):
        raise ValueError(f"Expected A with shape (12, 8), got {A.shape}")

    def fmt(x):
        if x is None or not np.isfinite(x) or np.isnan(x):
            return nan_token
        return f"{float(x):.{digits}f}"

    # Column spec and open
    lines = [r"\begin{tabular}{ccc|cc|cc|cc|cc}", r"\hline"]

    # Header row 1: group headings (bars after first 3 groups only)
    group_cells = []
    for i, n in enumerate(sample_sizes):
        bar = "|" if i < len(sample_sizes) - 1 else ""
        group_cells.append(rf"\multicolumn{{2}}{{c{bar}}}{{$n=N={n}$}}")
    lines.append(
        r"                           &                           &                  & "
        + " & ".join(group_cells)
        + r" \\ \hline"
    )

    # Header row 2: Mean/SD repeated
    lines.append(
        r"                           & $P$                       & OT map           & "
        + " & ".join(["Mean", "SD"] * len(sample_sizes))
        + r" \\ \hline"
    )

    # Pretty labels
    P_normal = r"$N(0,1)$"
    P_t      = rf"$t({t_df_label})$"
    OT_rank  = "Rank function"
    OT_signq = "Sign-quadratic"

    # Emit 3 estimator blocks × 4 rows each, with \cline{2-11} after row 2
    row = 0
    for est_name in estimator_names:
        # Row 1: estimator+P_normal multirows + Rank
        nums = " & ".join(fmt(v) for v in A[row]); row += 1
        lines.append(
            rf"\multirow{{4}}{{*}}{{{est_name}}}    & \multirow{{2}}{{*}}{{{P_normal}}} & {OT_rank}    & {nums} \\"
        )
        # Row 2: Signed-quadratic (within P_normal block)
        nums = " & ".join(fmt(v) for v in A[row]); row += 1
        lines.append(
            rf"                           &                           & {OT_signq} & {nums} \\ \cline{{2-11}}"
        )
        # Row 3: P_t multirow + Rank
        nums = " & ".join(fmt(v) for v in A[row]); row += 1
        lines.append(
            rf"                           & \multirow{{2}}{{*}}{{{P_t}}}   & {OT_rank}    & {nums} \\"
        )
        # Row 4: Signed-quadratic (within P_t block)
        nums = " & ".join(fmt(v) for v in A[row]); row += 1
        lines.append(
            rf"                           &                           & {OT_signq} & {nums} \\ \hline"
        )

    lines.append(r"\end{tabular}")
    return "\n".join(lines)


<>:70: SyntaxWarning: invalid escape sequence '\h'
<>:70: SyntaxWarning: invalid escape sequence '\h'
/var/folders/cn/mdbb7k_96272fvv6qp21ykp80000gn/T/ipykernel_63713/1229285088.py:70: SyntaxWarning: invalid escape sequence '\h'
  """


In [9]:
table_loss = summarize_dimension_table(
    rkhs_csv_dir,
    orig_roots[1],
    orig_roots[2],
    d_target=3,
    Ns=(100,300,500,1000),
    upper_clip=None,   # or set e.g. 50.0 to exclude extreme values
)

latex_table = table12x8_to_latex_with_clines(table_loss, digits=4)
print(latex_table)


\begin{tabular}{ccc|cc|cc|cc|cc}
\hline
                           &                           &                  & \multicolumn{2}{c|}{$n=N=100$} & \multicolumn{2}{c|}{$n=N=300$} & \multicolumn{2}{c|}{$n=N=500$} & \multicolumn{2}{c}{$n=N=1000$} \\ \hline
                           & $P$                       & OT map           & Mean & SD & Mean & SD & Mean & SD & Mean & SD \\ \hline
\multirow{4}{*}{Kernel}    & \multirow{2}{*}{$N(0,1)$} & Rank function    & 0.1118 & 0.0031 & 0.1082 & 0.0016 & 0.1072 & 0.0012 & 0.1067 & 0.0011 \\
                           &                           & Sign-quadratic & 1.2429 & 0.0437 & 1.1933 & 0.0237 & 1.1803 & 0.0180 & 1.1728 & 0.0168 \\ \cline{2-11}
                           & \multirow{2}{*}{$t(6)$}   & Rank function    & 0.1820 & 0.0083 & 0.1736 & 0.0049 & 0.1719 & 0.0041 & 0.1698 & 0.0033 \\
                           &                           & Sign-quadratic & 3.8209 & 0.4957 & 3.6073 & 0.4456 & 3.5996 & 0.4537 & 3.5169 & 0.4509 \\ \hline


In [ ]:
BOOTSTRAP_SEED = 20260527


def _median_bootstrap_stats(arr: np.ndarray, rng: np.random.Generator, n_boot: int = 2000) -> tuple[float, float]:
    if arr is None or arr.size == 0:
        return np.nan, np.nan
    vals = np.asarray(arr, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan, np.nan
    median = float(np.median(vals))
    if vals.size == 1:
        return median, np.nan
    idx = rng.integers(0, vals.size, size=(n_boot, vals.size))
    boot_medians = np.median(vals[idx], axis=1)
    boot_sd = float(np.std(boot_medians, ddof=1)) if n_boot > 1 else np.nan
    return median, boot_sd


def summarize_dimension_median_bootstrap_table(
    rkhs_root: str,
    path_nn1_root: str,
    path_nn2_root: str,
    d_target: int,
    Ns=(100, 300, 500, 1000),
    upper_clip: float | None = None,
    bootstrap_seed: int = BOOTSTRAP_SEED,
    n_boot: int = 2000,
) -> np.ndarray:
    """
    Same 12x8 layout as summarize_dimension_table, but each pair of columns is
    [median, bootstrap SD of the median]. Bootstrap RNG seed defaults to 20260527.
    """
    rkhs_map = load_rkhs_dir(rkhs_root)
    nn1_map = load_original_tree(path_nn1_root)
    nn2_map = load_original_tree(path_nn2_root)
    rng = np.random.default_rng(bootstrap_seed)

    cases = [
        ("normal", "CDF"),
        ("normal", "quadratic"),
        ("t", "CDF"),
        ("t", "quadratic"),
    ]
    estimators = [rkhs_map, nn1_map, nn2_map]
    rows = []

    for est_map in estimators:
        for measure, transform in cases:
            row = []
            for n in Ns:
                arr = est_map.get((measure, transform, int(n), int(d_target)), None)
                if arr is not None and upper_clip is not None:
                    arr = np.asarray(arr, dtype=float)
                    arr = arr[np.isfinite(arr) & (arr <= upper_clip)]
                median, boot_sd = _median_bootstrap_stats(arr, rng=rng, n_boot=n_boot)
                row.extend([median, boot_sd])
            rows.append(row)

    return np.array(rows, dtype=float)


table_loss_median = summarize_dimension_median_bootstrap_table(
    rkhs_csv_dir,
    orig_roots[1],
    orig_roots[2],
    d_target=3,
    Ns=(100, 300, 500, 1000),
    upper_clip=None,
    bootstrap_seed=BOOTSTRAP_SEED,
)

latex_median_table = table12x8_to_latex_with_clines(table_loss_median, digits=4)
latex_median_table = latex_median_table.replace("Mean", "Median").replace("SD", "Bootstrap SD")
print(latex_median_table)


In [70]:
from scipy.optimize import linprog

SETTINGS = [
    ("normal", "CDF"),        # N(0,1) - CDF (Rank)
    ("normal", "quadratic"),  # N(0,1) - Sign-quadratic
    ("t",      "CDF"),        # t(6)   - CDF (Rank)
    ("t",      "quadratic"),  # t(6)   - Sign-quadratic
]

def _collect_log_xy_for_setting(est_map, measure, transform, Ns, d_target, upper_clip=None):
    """
    Build (x, y) for quantile regression:
      x = log(sample size) repeated per replicate
      y = log(loss) for each replicate
    """
    xs, ys = [], []
    for n in Ns:
        arr = est_map.get((measure, transform, int(n), int(d_target)), None)
        if arr is None or len(arr) == 0:
            continue
        vals = np.asarray(arr, dtype=float)
        # keep only positive (log-safe) and optionally clip high values
        mask = vals > 0.0
        if upper_clip is not None:
            mask &= (vals <= upper_clip)
        vals = vals[mask]
        if vals.size == 0:
            continue
        ys.append(np.log(vals))
        xs.append(np.full(vals.shape, np.log(n), dtype=float))
    if not xs:
        return None, None
    return np.concatenate(xs), np.concatenate(ys)

def _quantile_slope_lp(x, y, tau=0.5):
    """
    Solve Koenker–Bassett quantile regression with a single predictor:
      min_{beta0,beta1, u,v >= 0}  tau*1^T u + (1-tau)*1^T v
      s.t.  X beta + u - v = y,  X=[1, x]
    Return beta1 (slope). NaN on failure/insufficient data.
    """
    if x is None or y is None or x.size < 2:
        return np.nan
    m = x.size
    X = np.column_stack([np.ones(m), x])  # m×2
    # Variables: [beta0, beta1, u(1..m), v(1..m)]  -> total 2 + 2m
    c = np.concatenate([np.zeros(2), tau*np.ones(m), (1.0 - tau)*np.ones(m)])
    Aeq = np.concatenate([X, np.eye(m), -np.eye(m)], axis=1)
    beq = y
    bounds = [(None, None), (None, None)] + [(0, None)]*m + [(0, None)]*m
    res = linprog(c, A_eq=Aeq, b_eq=beq, bounds=bounds, method="highs")
    if not res.success:
        return np.nan
    beta = res.x[:2]
    return float(beta[1])

def quantile_slopes_three_estimators(
    rkhs_root: str,
    nn1_root: str,
    nn2_root: str,
    d_target: int,
    sample_sizes=(100, 300, 500, 1000),
    tau: float = 0.50,
    upper_clip: float | None = None,
) -> np.ndarray:
    """
    Fit log-loss ~ intercept + slope*log(n) at quantile `tau` for each estimator × setting.

    Returns:
      slopes : (3, 4) float array
               rows = ["Kernel", "nn_1", "nn_2"]
               cols = ["N(0,1)-CDF", "N(0,1)-sign quadratic",
                       "t(6)-CDF", "t(6)-sign quadratic"]
    """
    rkhs_map = load_rkhs_dir(rkhs_root)
    nn1_map  = load_original_tree(nn1_root)
    nn2_map  = load_original_tree(nn2_root)

    est_maps = [rkhs_map, nn1_map, nn2_map]
    slopes = np.full((3, 4), np.nan, dtype=float)

    for ei, est_map in enumerate(est_maps):
        for si, (measure, transform) in enumerate(SETTINGS):
            x, y = _collect_log_xy_for_setting(est_map, measure, transform,
                                               sample_sizes, d_target, upper_clip)
            slopes[ei, si] = _quantile_slope_lp(x, y, tau=tau)
    return slopes

def collect_slopes_matrix(
    rkhs_root: str,
    nn1_root: str,
    nn2_root: str,
    dims=(3, 5, 10),
    sample_sizes=(100, 300, 500, 1000),
    tau: float = 0.50,
    upper_clip: float | None = None,
) -> np.ndarray:
    """
    Build a 9x4 matrix of slopes using quantile_slopes_three_estimators.

    Row order (9 rows):
      0..2: Kernel slopes for d in dims (in order)
      3..5: nn_1   slopes for d in dims
      6..8: nn_2   slopes for d in dims

    Columns (4):
      [N(0,1)-CDF, N(0,1)-sign quadratic, t(6)-CDF, t(6)-sign quadratic]
    """
    dims = list(dims)
    results_by_d = {}
    for d in dims:
        S = quantile_slopes_three_estimators(
            rkhs_root=rkhs_root,
            nn1_root=nn1_root,
            nn2_root=nn2_root,
            d_target=d,
            sample_sizes=sample_sizes,
            tau=tau,
            upper_clip=upper_clip,
        )  # shape (3,4): rows = [Kernel, nn_1, nn_2]
        results_by_d[d] = S

    rows = []
    # Kernel rows across dims
    for d in dims:
        rows.append(results_by_d[d][0, :])
    # nn_1 rows across dims
    for d in dims:
        rows.append(results_by_d[d][1, :])
    # nn_2 rows across dims
    for d in dims:
        rows.append(results_by_d[d][2, :])

    return np.vstack(rows)  # shape (9, 4)


def slopes_to_latex_compact_with_dim_blocks(
    slopes: np.ndarray,
    dims_per_est: Sequence[int] = (3, 5, 10),
    estimator_names: Sequence[str] = ("Kernel", "ICNN-ReLU", "ICNN-ELU"),
    digits: int = 4,
    t_df_label: int = 6,
    nan_token: str = "--",
) -> str:
    """
    Produce LaTeX like:

    \begin{tabular}{c|c|cc|cc}
    \hline
                               &     & \multicolumn{2}{c|}{$N(0,1)$} & \multicolumn{2}{c}{$t(6)$} \\ \hline
    Estimator                  & $d$ & CDF         & Sign-quadratic    & CDF       & Sign-quadratic   \\ \hline
    \multirow{3}{*}{Kernel}    & 3   & ... \\ 
                               & 5   & ... \\
                               & 10  & ... \\ \hline
    \multirow{3}{*}{ICNN-ReLU} & 3   & ... \\
    ...
    \end{tabular}

    Input
    -----
    slopes : array, shape (E*D, 4)
        Rows ordered by estimator then dimension.
        Cols: [N(0,1)-CDF, N(0,1)-Sign-quadratic, t(df)-CDF, t(df)-Sign-quadratic]
    dims_per_est : iterable of the D dimensions per estimator block (e.g., (3,5,10))
    estimator_names : length E (e.g., ("Kernel","ICNN-ReLU","ICNN-ELU"))
    digits : number of decimals to print
    """
    A = np.asarray(slopes, dtype=float)
    E = len(estimator_names)
    D = len(dims_per_est)
    if A.shape != (E * D, 4):
        raise ValueError(f"Expected slopes with shape ({E*D}, 4), got {A.shape}")

    def fmt(x):
        if x is None or not np.isfinite(x) or np.isnan(x):
            return nan_token
        return f"{float(x):.{digits}f}"

    lines = []
    lines.append(r"\begin{tabular}{c|c|cc|cc}")
    lines.append(r"\hline")
    lines.append(
        rf"                           &     & \multicolumn{{2}}{{c|}}{{${{N(0,1)}}$}} & "
        rf"\multicolumn{{2}}{{c}}{{${{t({t_df_label})}}$}} \\ \hline"
    )
    lines.append(r"Estimator                  & $d$ & CDF         & Sign-quadratic    & CDF       & Sign-quadratic   \\ \hline")

    row_idx = 0
    for est in estimator_names:
        # First row of the block: include \multirow{D}{*}{est}
        d = dims_per_est[0]
        vals = [fmt(v) for v in A[row_idx]]
        lines.append(
            rf"\multirow{{{D}}}{{*}}{{{est}}}    & {d}   & {vals[0]}     & {vals[1]}           & {vals[2]}   & {vals[3]}          \\"
        )
        row_idx += 1
        # Middle rows (2..D): blank estimator cell
        for k in range(1, D):
            d = dims_per_est[k]
            vals = [fmt(v) for v in A[row_idx]]
            lines.append(
                rf"                           & {d}   & {vals[0]}     & {vals[1]}           & {vals[2]}   & {vals[3]}          \\"
            )
            row_idx += 1
        # Block separator
        lines.append(r"\hline")

    lines.append(r"\end{tabular}")
    return "\n".join(lines)


<>:145: SyntaxWarning: invalid escape sequence '\h'
<>:145: SyntaxWarning: invalid escape sequence '\h'
/var/folders/cn/mdbb7k_96272fvv6qp21ykp80000gn/T/ipykernel_11082/2208348302.py:145: SyntaxWarning: invalid escape sequence '\h'
  """


In [72]:
M = collect_slopes_matrix(
    rkhs_root="ot_rkhs_rmse_results",
    nn1_root="ICNN_relu",
    nn2_root="ICNN_elu",
    dims=[3,5,10],
    sample_sizes=[300,500,1000],
    tau=0.5,
    upper_clip=100.0
)

dims_block = (3,5,10)
latex = slopes_to_latex_compact_with_dim_blocks(
    M, dims_per_est=dims_block, estimator_names=("Kernel","ICNN-ReLU","ICNN-ELU"), digits=4, t_df_label=6
)
print(latex)


\begin{tabular}{c|c|cc|cc}
\hline
                           &     & \multicolumn{2}{c|}{${N(0,1)}$} & \multicolumn{2}{c}{${t(6)}$} \\ \hline
Estimator                  & $d$ & CDF         & Sign-quadratic    & CDF       & Sign-quadratic   \\ \hline
\multirow{3}{*}{Kernel}    & 3   & -0.0098     & -0.0114           & -0.0159   & -0.0165          \\
                           & 5   & -0.0160     & -0.0161           & -0.0214   & -0.0272          \\
                           & 10   & -0.0254     & -0.0271           & -0.0297   & -0.0286          \\
\hline
\multirow{3}{*}{ICNN-ReLU}    & 3   & -0.2311     & -0.1217           & -0.2537   & -0.1191          \\
                           & 5   & -0.2318     & -0.2018           & -0.2208   & -0.3327          \\
                           & 10   & -0.1850     & -0.1800           & -0.2154   & -0.2562          \\
\hline
\multirow{3}{*}{ICNN-ELU}    & 3   & -0.3939     & -0.2722           & -0.4033   & -0.3636          \\
                      